In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter

In [ ]:
# 1. LOAD CLEAN DATA
df = pd.read_csv('../data/clean_data.csv')
print("Data Loaded. Shape:", df.shape)

Data Loaded. Shape: (9976, 55)


In [ ]:
# 2. FEATURE SELECTION & CLEANING
# CRITICAL: Drop the target text column 'loan_status' BEFORE encoding.
df = df.drop(columns=['loan_status'])

In [10]:
# Drop other high-cardinality/unnecessary columns.
# 'sub_grade' is redundant (we have 'grade'), and 'issue_month' is not for this model.
cols_to_drop = ['sub_grade', 'issue_month', 'balance', 'paid_total', 
                 'paid_principal', 'paid_interest', 'paid_late_fees']
df = df.drop(columns=cols_to_drop)

In [11]:
# 3. ONE-HOT ENCODING
# Encode remaining categorical variables (like 'grade', 'homeownership', 'state')
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Encoding features: {categorical_cols.tolist()}")

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", df_encoded.shape)

Encoding features: ['state', 'homeownership', 'verified_income', 'verification_income_joint', 'loan_purpose', 'application_type', 'grade', 'initial_listing_status', 'disbursement_method']
Shape after encoding: (9976, 114)


In [ ]:
# 4. DEFINE X and y
# y is the binary target (0 or 1)
y = df_encoded['loan_status_binary']

# X is everything else (drop the target)
X = df_encoded.drop(columns=['loan_status_binary'])

In [ ]:
# --- HANDLE REMAINING NANS ---
# Check if any NaNs exist
if X.isnull().sum().sum() > 0:
    print(f"\nWARNING: Found {X.isnull().sum().sum()} missing values in X. Filling them now...")
    # Fill numeric columns with median (robust to outliers)
    X = X.fillna(X.median())
    
    # Double check
    print(f"Missing values after fill: {X.isnull().sum().sum()}")
else:
    print("\nNo missing values found in X. Proceeding...")


Missing values after fill: 0


In [14]:
# 5. TRAIN - TEST SPLIT
# Stratify ensures we keep the same % of bad loans in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Original Train class distribution: {Counter(y_train)}")


Training set shape: (6983, 113)
Test set shape: (2993, 113)
Original Train class distribution: Counter({0: 6859, 1: 124})


In [15]:
# 6. APPLY SMOTE (Fixing Class Imbalance)
# This creates fake "Bad" loans so the model isn't biased toward "Good" loans
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nResampled Train class distribution: {Counter(y_train_resampled)}")


Resampled Train class distribution: Counter({0: 6859, 1: 6859})


In [17]:
# 6. APPLY SMOTE
print("\nApplying SMOTE...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled Train class distribution: {Counter(y_train_resampled)}")

# 7. SAVE PROCESSED DATA
X_train_resampled.to_csv('../data/X_train_resampled.csv', index=False)
y_train_resampled.to_csv('../data/y_train_resampled.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("\nSUCCESS: Feature Engineering Complete. Data saved for modeling!")


Applying SMOTE...
Resampled Train class distribution: Counter({0: 6859, 1: 6859})

SUCCESS: Feature Engineering Complete. Data saved for modeling!
